# **DURABILITY — $\lambda$ MAPS**

Scatter of every design point at a chosen time step: one design variable on $x$, another on $y$,
$\lambda_i$ on the color bar — one **separate** figure per $\lambda_i$ ($i=1,\dots,4$), each saved
on its own so they can be arranged freely in Overleaf. No title on the figures themselves — that
belongs in the LaTeX caption.

Loads `dataset_unique_<split>_<tag>.pkl` from
[`01_generate_dataset.ipynb`](01_generate_dataset.ipynb) — **not** `dataset_full`: `dataset_full`
has one row per *latent* draw (thousands per design point, all sharing the same design point and
$\lambda$), so plotting it would just redraw the same dot on top of itself thousands of times for
no visual gain. `dataset_unique` already has exactly one row per design point.

The durability problem has **three** design variables (`fck`, `rh`, `cov`), but a scatter only has
room for two spatial axes plus color — pick the pair with `x_var`/`y_var` below; the third variable
is simply not shown (it still varies across the points plotted, it's just not encoded visually).

CO₂ uses the published CMIP6/SSP table. Set `installation_year` and `co2_scenario` below.
The 100-year horizon must end no later than 2100. Regenerate datasets and retrain after changing the scenario; old polynomial results are not compatible.


## 1. Libraries

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update({
                        'font.family': 'serif',
                        'mathtext.fontset': 'cm',
                        'axes.unicode_minus': False
                    })
import numpy as np
import pandas as pd

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import dill

## 2. Config

`n_latent_samples`, `times`, `installation_year`, `co2_scenario`, `cement_type` and `exposure_conditions` must
match [`01_generate_dataset.ipynb`](01_generate_dataset.ipynb) — together they name the file being
loaded. Pick the time step with `time_index` (its position in `times`) rather than typing the float
directly, to avoid a mismatch against the saved filename.

`x_var`/`y_var` choose which 2 of the 3 design variables (`'fck'`, `'rh'`, `'cov'`) go on the
scatter's axes.

`xlim`/`ylim` and `lambda_vlim` are `None` by default (matplotlib auto-scales each figure to its
own data). Set them explicitly to force the same axis range / color range across figures — e.g.
across different `time_index` values, so the panels are visually comparable in the paper.

In [2]:
n_latent_samples     = 2500     # must match stage 1 — it names the file being loaded
times                = np.linspace(0, 100, 5, endpoint=True)  # must match stage 1
installation_year    = 1980     # must match stage 1
co2_scenario         = "SSP2-4.5"  # SSP1-2.6, SSP2-4.5, or SSP5-8.5
cement_type          = 3        # must match stage 1
exposure_conditions  = 2        # must match stage 1

split       = 'train'   # 'train' or 'val'
time_index  = 2          # index into `times` — change this to look at a different time step
time_step   = times[time_index]

fig_size   = (5, 4)      # size of each individual figure, in inches
fig_format = 'png'       # format each figure is saved in ('pdf', 'png', ...)
fig_dpi    = 300         # resolution the figure is saved at (dots per inch)

label_fontsize = 14   # font size of the axis labels and colorbar label
tick_fontsize  = 12   # font size of the tick numbers, on both axes and the colorbar

# Per-lambda colorbar limits as (vmin, vmax). None = auto-scaled to that lambda's own range.
lambda_vlim = {
                'lambda 1': None,
                'lambda 2': None,
                'lambda 3': None,
                'lambda 4': None,
              }

print(f"Plotting t = {time_step:.2f} years ({split} split)")

Plotting t = 50.00 years (train split)


## 3. Load the dataset

In [3]:
tag      = f'{time_step}_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}_co2_{co2_scenario}'
pkl_name = f'{n_latent_samples}_dataset_unique_{split}_{tag}'

with open(f'{pkl_name}.pkl', 'rb') as f:
    df = dill.load(f)

print(f"{len(df)} design points loaded")
df.head()

200 design points loaded


         fck         rh        cov  ...  lambda 3  lambda 4  Processing time (s)
0  35.146120  56.813984  19.354691  ...  0.109717  0.159715             0.102160
1  44.305805  75.929459  26.259503  ...  0.131995  0.165475             0.146034
2  33.286039  21.911130  51.431472  ...  0.132796  0.140462             0.144206
3  31.794235  41.076697  22.899861  ...  0.110537  0.141987             0.094516
4  20.294744  61.405519  23.505869  ...  0.093325  0.132580             0.109834

[5 rows x 8 columns]

## 4. $\lambda$ maps (3D)

The durability problem has three design variables (`fck`, `rh`, `cov`); a 2D scatter would have to
drop one of them, so this puts all three on the 3 spatial axes together and keeps $\lambda_i$ on
the color bar — one figure per $\lambda_i$, no title (captions belong in the LaTeX), configurable
size/fonts/dpi, saved next to the `.pkl`. `elev`/`azim` set the camera angle.

In [4]:
var_labels = {'fck': '$f_{ck}$ (MPa)', 'rh': '$RH$ (%)', 'cov': '$c$ (mm)'}
lambda_cols = ['lambda 1', 'lambda 2', 'lambda 3', 'lambda 4']

fig_size_3d = (7, 6)   # 3D plots need more room than a 2D one, for the extra axis + colorbar
elev = 20   # camera elevation angle, in degrees
azim = -60  # camera azimuth angle, in degrees

figs_3d = {}

for col in lambda_cols:
    mask       = df[col].notna()
    vmin, vmax = lambda_vlim.get(col) or (None, None)
    idx        = col.split(' ')[1]

    fig = plt.figure(figsize=fig_size_3d)
    ax  = fig.add_subplot(projection='3d')
    sc = ax.scatter(df.loc[mask, 'fck'], df.loc[mask, 'rh'], df.loc[mask, 'cov'], c=df.loc[mask, col],
                     cmap='coolwarm', s=20, edgecolor='0.3', linewidth=0.2, vmin=vmin, vmax=vmax)
    cbar = fig.colorbar(sc, ax=ax, shrink=0.6, pad=0.15)
    cbar.set_label(f'$\lambda_{{{idx}}}$', fontsize=label_fontsize)
    cbar.ax.tick_params(labelsize=tick_fontsize)
    ax.set_xlabel(var_labels['fck'], fontsize=label_fontsize, labelpad=15)
    ax.set_ylabel(var_labels['rh'], fontsize=label_fontsize, labelpad=15)
    ax.set_zlabel(var_labels['cov'], fontsize=label_fontsize, labelpad=15)
    ax.tick_params(axis='both', labelsize=tick_fontsize)
    ax.view_init(elev=elev, azim=azim)

    fig.savefig(f'{pkl_name}_lambda_{idx}.{fig_format}', bbox_inches='tight', dpi=fig_dpi)
    figs_3d[col] = fig
    plt.show()